# 01 — Dataset preparation

ASHA-AI Plan 2.0 · Role C — AI / ML / Voice Lead

Builds the training dataset Member B's backend consumes via `xgboost_v1.pkl`. The shared implementation lives in [`../train_and_eval.py`](../train_and_eval.py) so notebooks and the CLI runner stay in lock-step.

**Source-of-truth precedence:**
1. `ml/datasets/disease_symptom_dataset.csv` (real Kaggle Disease-Symptom Prediction) + `ml/datasets/disease_to_care_v1.json` mapping — preferred.
2. Otherwise: rule-grounded synthetic set (~4 500 rows) generated from the 9 red-flag templates + 15 clinic + 7 home-care presentations.

Output: `ml/datasets/synthetic_train_v1.csv` (or `kaggle_train_v1.csv`).


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..').resolve()))
from train_and_eval import build_training_dataset, ML, SEVERITY, SYMPTOMS, feature_columns
print(f'Vocabulary size: {len(SEVERITY)} symptoms')
print(f'Total feature columns: {len(feature_columns())}')


In [ ]:
df = build_training_dataset(n_per_class=1500, seed=42)
print(f'rows={len(df)}')
df['care_level'].value_counts()


In [ ]:
from collections import Counter
import pandas as pd
# Sanity-check the top symptom co-occurrences per class — these should reflect the rule structure.
for klass in ['Emergency Room', 'Clinic Visit', 'Home Care']:
    sub = df[df.care_level == klass]
    sym_counts = sub.filter(like='sym_').sum().sort_values(ascending=False).head(8)
    print(f'\n--- {klass} (n={len(sub)}) top symptoms ---')
    print(sym_counts.to_string())


In [ ]:
(ML / 'datasets').mkdir(parents=True, exist_ok=True)
df.to_csv(ML / 'datasets' / 'synthetic_train_v1.csv', index=False)
print('Wrote', ML / 'datasets' / 'synthetic_train_v1.csv')


### Plan 2.0 swap-in: real Kaggle dataset

When the team downloads the Kaggle Disease-Symptom Prediction dataset (Itachi9604):

```bash
# from your Kaggle account:
kaggle datasets download -d itachi9604/disease-symptom-description-dataset \
    -p ml/datasets/ --unzip
```

then author the disease→care mapping at `ml/datasets/disease_to_care_v1.json`. `build_training_dataset` will pick it up automatically (see `_build_from_kaggle` in `train_and_eval.py`). No code change needed beyond the mapping JSON.